# Tutorial 0: Introduction to Continual Learning

This tutorial introduces the core problem that continual learning solves: **catastrophic forgetting** in neural networks. You will see forgetting happen live in a simple MLP, learn how to measure it, and get a preview of the five strategies this course teaches for fighting it.

## What is Continual Learning?

**Continual learning** (also called lifelong learning or incremental learning) is the ability of a machine learning model to learn new knowledge over time without forgetting what it already knows.

This sounds simple, but it turns out to be one of the hardest open problems in deep learning. The root cause is a fundamental tension in how neural networks learn:

- Neural networks store knowledge **distributed across their weights**. There is no address book — knowledge about cats, grammar, and arithmetic is all entangled in the same weight matrices.
- When you train on new data, **gradient descent pushes weights toward the new objective**, and the old knowledge encoded in those weights gets overwritten.
- This is called **catastrophic forgetting**, and it is not a bug in any particular architecture — it is a fundamental property of gradient-based optimization with shared parameters.

### Real-world analogy

Imagine studying intensively for a biology exam for two weeks. You ace it. Then you spend two weeks studying nothing but calculus. When someone asks you a biology question, your recall has degraded — not because the knowledge was never there, but because your brain's working memory has been overwritten by calculus patterns.

Neural networks experience this same phenomenon, but **far more severely**. A model fine-tuned on medical text can lose its ability to write Python code. A chatbot trained on customer support data can forget how to do basic math.

### Why this matters for LLMs in production

Large Language Models deployed in production need to:

1. **Absorb new information** — product updates, policy changes, breaking news, new regulations
2. **Retain general capabilities** — reasoning, language understanding, coding, domain expertise
3. **Adapt without full retraining** — which can cost millions of dollars and weeks of GPU-hours

The five strategies in this course each offer a different solution to this problem, ranging from surgical weight modifications to approaches that never touch the weights at all.

## The Catastrophic Forgetting Problem

When you fine-tune a neural network on new data, the gradients push weights toward configurations that minimize loss on the **new** data. But the old knowledge was encoded in those same weights — and the new gradient updates overwrite it.

### Weight space visualization

```
     Loss landscape (simplified 2D projection)

          Task A optimum        Task B optimum
               *                     *
              / \                   / \
             /   \                 /   \
            /     \    gradient   /     \
           /       \   descent  /       \
          /         \ -------> /         \
         /           \       /           \
        /    low loss \     / low loss    \
       /   for Task A  \   / for Task B    \
      /                 \ /                 \
     /                   X                   \
    /              high loss for               \
   /               BOTH tasks                   \
```

The core challenge: **Task A's optimal weights and Task B's optimal weights occupy different regions of weight space.** Moving toward one means moving away from the other.

Continual learning strategies attack this problem in different ways:
- **Regularization**: Penalize changes to important weights (EWC, SI)
- **Replay**: Mix old data with new data during training
- **Architecture**: Use separate parameters for each task (DualMLP, LoRA)
- **Retrieval**: Don't modify weights at all — retrieve knowledge at inference time (JitRL, ACE)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Create a simple 2-layer MLP
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 32)
        self.fc2 = nn.Linear(32, 2)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

# Task A: Classify points in upper-right quadrant
torch.manual_seed(42)
task_a_x = torch.randn(200, 2)
task_a_y = ((task_a_x[:, 0] > 0) & (task_a_x[:, 1] > 0)).long()

# Task B: Classify points by x > 0
task_b_x = torch.randn(200, 2)
task_b_y = (task_b_x[:, 0] > 0).long()

model = TinyMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train on Task A
for epoch in range(100):
    loss = F.cross_entropy(model(task_a_x), task_a_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

acc_a_before = (model(task_a_x).argmax(1) == task_a_y).float().mean().item()
print(f"Task A accuracy (after training on A): {acc_a_before:.2%}")

# Now train on Task B
for epoch in range(100):
    loss = F.cross_entropy(model(task_b_x), task_b_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

acc_a_after = (model(task_a_x).argmax(1) == task_a_y).float().mean().item()
acc_b = (model(task_b_x).argmax(1) == task_b_y).float().mean().item()
forgetting = (acc_a_before - acc_a_after) / acc_a_before if acc_a_before > 0 else 0

print(f"\nTask A accuracy (after training on B): {acc_a_after:.2%}")
print(f"Task B accuracy: {acc_b:.2%}")
print(f"Forgetting ratio: {forgetting:.2%}")
print(f"\n{'CATASTROPHIC FORGETTING!' if forgetting > 0.15 else 'Minimal forgetting'}")

## Measuring Forgetting

The **forgetting ratio** is the standard metric for quantifying how much old knowledge a model has lost after learning something new:

$$\text{forgetting\_ratio} = \frac{\text{accuracy\_before} - \text{accuracy\_after}}{\text{accuracy\_before}}$$

| Value | Interpretation |
|-------|---------------|
| **0.0** | No forgetting — the model retained all old knowledge perfectly |
| **0.15** | 15% degradation — the threshold we target in this project |
| **0.50** | 50% degradation — the model has lost half its old knowledge |
| **1.0** | Total forgetting — old knowledge is completely gone |
| **< 0** | Negative forgetting — the model actually *improved* on old tasks (rare but possible via positive transfer) |

**Our target: forgetting ratio < 0.15** (less than 15% degradation on previously learned tasks).

The demo above likely showed a forgetting ratio well above 0.15 — this is the problem we are solving.

## Five Strategies to Fight Forgetting

This course teaches five distinct strategies, each with different tradeoffs:

| Strategy | Weight Updates? | Learn Speed | Forgetting Risk | Best For |
|----------|----------------|-------------|-----------------|----------|
| **TTT-E2E** | Yes (trainable MLP) | ~12s | Low | Deep knowledge internalization |
| **JitRL MVP** | No | ~0.002s | Zero | Fast QA, simple retrieval |
| **JitRL Full** | No | ~0.04s | Zero | Semantic retrieval tasks |
| **ACE** | No | ~30s | Zero | Strategy evolution, meta-learning |
| **Doc-to-LoRA** | Yes (LoRA) | ~0.8s | Very low | Single-pass document learning |

The most fundamental distinction is whether a strategy **modifies the model's weights**:

- **Weight-modifying** (TTT-E2E, Doc-to-LoRA): Deeper knowledge integration, but must manage forgetting risk
- **Non-weight-modifying** (JitRL MVP, JitRL Full, ACE): Zero forgetting by design, but knowledge is external to the model

## Strategy 1 Preview: TTT-E2E (Test-Time Training End-to-End)

**Key insight**: Instead of modifying existing MLP layers, add a *second* trainable MLP alongside each frozen one.

The final 25% of transformer layers (layers 21-28 in Qwen2.5-1.5B) are modified with a **Dual-MLP architecture**:
- **Frozen MLP**: The original weights, locked and untouched — preserving general intelligence
- **Trainable MLP**: A new copy that absorbs new knowledge via gradient descent
- **Alpha blending**: `output = frozen_out + (1 - alpha) * trainable_out` — alpha starts near 1.0 and decays as documents are learned

**TF-IDF gating** adds a second layer of protection: neurons that fire broadly across a calibration corpus are masked during gradient updates, so only document-specific neurons receive gradients.

Result: the model can learn new facts while keeping its general reasoning intact.

> **Deep dive**: Tutorial 1 walks through the full DualMLP architecture, TF-IDF calibration, and alpha decay schedule.

## Strategy 2 Preview: JitRL MVP (Just-in-Time Retrieval Learning)

**Key insight**: Don't modify weights at all — retrieve relevant passages at query time and bias the model's outputs toward them.

The MVP engine works in two phases:
1. **Learn**: Chunk the document and build a TF-IDF index over the chunks
2. **Generate**: At query time, retrieve the top-3 most relevant chunks, prepend them as context, and apply **logit biasing** to nudge the model toward tokens found in retrieved content

Since no parameters are updated, there is **zero forgetting risk**. The tradeoff is that knowledge is limited to what fits in the context window.

> **Deep dive**: Tutorial 2 builds the TF-IDF retriever and logit biaser from scratch.

## Strategy 3 Preview: JitRL Full (Reward-Guided Logit Modulation)

**Key insight**: Encode documents into hidden-state embeddings and use reward signals to modulate generation.

JitRL Full is more sophisticated than MVP:
1. **Learn**: Pass the document through the model and capture the last hidden-state embeddings. Store them in a **knowledge store**.
2. **Generate**: At query time, retrieve the closest knowledge embeddings via cosine similarity. Compute a **reward signal** and modulate the model's logit distribution to favor knowledge-aligned tokens.

Still zero weight updates, but the embedding-based retrieval captures richer semantic relationships than TF-IDF.

> **Deep dive**: Tutorial 3 implements the knowledge store, reward computation, and logit modulation pipeline.

## Strategy 4 Preview: ACE (Agentic Context Engineering)

**Key insight**: Instead of modifying the model or its retrieval index, evolve the *prompting strategy* through LLM-powered self-improvement loops.

ACE runs three roles in a loop:
- **Generator**: Answers questions using the document and current playbook strategies
- **Reflector**: Critiques the answer — what went right, what went wrong, suggested improvements
- **Curator**: Patch-updates the playbook with minimal, targeted changes (never a full rewrite)

Each cycle makes the playbook smarter: failures become strategies, successes become rules. Playbooks persist as JSON files across sessions.

Requires Ollama running locally with a capable model (e.g., `qwen3.5:9b`).

> **Deep dive**: Tutorial 4 builds the Generate-Reflect-Curate loop and shows playbook evolution across multiple learning cycles.

## Strategy 5 Preview: Doc-to-LoRA (Hypernetwork-Generated Adapters)

**Key insight**: A hypernetwork reads document text and outputs LoRA weight matrices in a single forward pass — no gradient descent needed.

The pipeline:
1. **Chunk** the document into 1024-token segments
2. **Extract activations** by passing chunks through the frozen base model
3. **Generate LoRA matrices** via a Perceiver-based hypernetwork (8 cross-attention blocks)
4. **Inject** the rank-8 LoRA adapters into the model via peft

Two modes are supported:
- **Doc mode**: Document content becomes LoRA adapters encoding that content
- **Text mode**: A task description (e.g., "answer questions about quantum physics") becomes task-specialized LoRA adapters

Based on Sakana AI research: [Doc-to-LoRA](https://arxiv.org/abs/2602.15902) and [Text-to-LoRA](https://arxiv.org/abs/2506.06105).

> **Deep dive**: Tutorial 5 walks through the hypernetwork architecture, LoRA injection, and both doc/text modes.

In [ ]:
import torch
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# Test key imports
try:
    from continual_learning.model.dual_mlp import DualMLP
    from continual_learning.jitrl.mvp.engine import JitRLMVPEngine
    from continual_learning.doc2lora.engine import Doc2LoRAEngine
    print("\nAll imports OK!")
except ImportError as e:
    print(f"\nImport error: {e}")
    print("Run: pip install -e . from the project root")

## What's Next

Now that you understand the forgetting problem and have seen it in action, proceed through the tutorials:

| Tutorial | Topic | What You Will Learn |
|----------|-------|---------------------|
| [Tutorial 1: TTT-E2E](01_ttt_e2e.ipynb) | Test-Time Training | DualMLP injection, TF-IDF gradient masking, alpha decay schedule |
| [Tutorial 2: JitRL MVP](02_jitrl_mvp.ipynb) | Retrieval-Augmented Learning | TF-IDF retrieval, logit biasing, zero-weight-update learning |
| [Tutorial 3: JitRL Full](03_jitrl_full.ipynb) | Reward-Guided Modulation | Hidden-state embeddings, knowledge store, reward signals |
| [Tutorial 4: ACE](04_ace.ipynb) | Agentic Context Engineering | Generate-Reflect-Curate loops, playbook evolution, Ollama integration |
| [Tutorial 5: Doc-to-LoRA](05_doc2lora.ipynb) | Hypernetwork Adapters | LoRA generation, Perceiver architecture, doc/text modes |
| [Tutorial 6: Benchmark](06_benchmark.ipynb) | Strategy Comparison | Side-by-side evaluation of all five strategies on identical data |